# Generate SPFC on T2I-CompBench 100 (Kaggle)

Runs only Sparse Primitive Flow Composition for the checked-in 100-prompt T2I-CompBench subset.

Outputs are written under `/kaggle/working/t2i_compbench_seed13/runs/t2i_compbench/spfc` and persist as this notebook's Kaggle output dataset.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Soobiwan/aim-flow.git'

%cd /kaggle/working
!rm -rf /kaggle/working/aim-flow
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
from pathlib import Path
import json
import os
import shutil
import shlex
import subprocess
import sys
import time

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("DIFFUSERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / RUN_SLUG
RUN_ROOT = OUTPUT_ROOT / "runs"
MANIFEST_REL = Path("configs/t2i_compbench_100_seed13.json")
DECOMP_REL = Path("configs/t2i_compbench_100_seed13_spfc.json")
RECTIFIED_REPO_DIR = WORK_ROOT / "Rectified-CFGpp"
INSTALL_DEPS = True

# If your repo is attached with a different Kaggle dataset slug, this auto-discovers it under /kaggle/input.
def has_aim_flow_repo(path: Path) -> bool:
    return (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_generate.py").exists()


def find_aim_flow_repo() -> Path | None:
    cwd = Path.cwd()
    if has_aim_flow_repo(cwd):
        return cwd
    dest = WORK_ROOT / "aim-flow"
    if has_aim_flow_repo(dest):
        return dest
    input_root = Path("/kaggle/input")
    candidates = [input_root / "aim-flow", input_root / "aim-flow" / "aim-flow"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "aim-flow"])
    for candidate in candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    return None


def ensure_working_repo() -> Path:
    source = find_aim_flow_repo()
    if source is None:
        raise FileNotFoundError(
            "Could not find aim-flow. Attach the repo as a Kaggle dataset, clone it into /kaggle/working, "
            "or run this notebook from the repo root."
        )
    dest = WORK_ROOT / "aim-flow" if Path("/kaggle").exists() else source
    if source.resolve() != dest.resolve():
        shutil.copytree(source, dest, dirs_exist_ok=True)
        return dest
    return source


def run_args(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", shlex.join([str(arg) for arg in args]))
    started = time.time()
    result = subprocess.run([str(arg) for arg in args], cwd=str(cwd) if cwd else None, text=True)
    print(f"elapsed: {(time.time() - started) / 60:.2f} min")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


REPO_DIR = ensure_working_repo()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)
print("outputs:", OUTPUT_ROOT)

if INSTALL_DEPS:
    run_args([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=REPO_DIR)
    run_args([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=REPO_DIR)

In [ ]:
METHOD = "spfc"
METHOD_TITLE = "SPFC"
GUIDANCE_SCALE = 4.5
print(f"Generating {METHOD_TITLE} with method={METHOD}, seed={SEED}, guidance_scale={GUIDANCE_SCALE}")

In [ ]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
HF_SECRET_NAMES = ('Huggingface', 'HF_TOKEN', 'HUGGINGFACE_TOKEN')

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in HF_SECRET_NAMES:
            try:
                token = secrets.get_secret(secret_name)
            except Exception:
                token = None
            if token:
                os.environ['HF_TOKEN'] = token
                break
    except Exception:
        pass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        'Missing Hugging Face token. In Kaggle, add a secret named Huggingface or HF_TOKEN, '
        'turn it on for this notebook, and make sure that Hugging Face account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF_TOKEN is set, but access check for {MODEL_ID} failed. '
        'Confirm the Kaggle secret is enabled and the token account has accepted the gated model license.'
    ) from exc

display(Markdown('Hugging Face token is configured and can access SD3 Medium.'))


run_args(["nvidia-smi"], check=False)

In [ ]:
# If Kaggle gives a P100 with an incompatible Torch build, uncomment these and restart the runtime.
!pip uninstall -q -y torch torchvision torchaudio
!pip install -q --no-cache-dir --force-reinstall torch==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -e .


In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

sample_count = len(manifest["samples"])
print("manifest:", manifest_path)
print("benchmark:", manifest["benchmark"])
print("samples:", sample_count)
assert sample_count == 100, f"Expected 100 T2I-CompBench samples, found {sample_count}."

if METHOD == "spfc":
    decomp_path = REPO_DIR / DECOMP_REL
    if not decomp_path.exists():
        raise FileNotFoundError(decomp_path)
    with decomp_path.open("r", encoding="utf-8") as f:
        decompositions = json.load(f)
    print("decompositions:", decomp_path)
    assert len(decompositions["items"]) == 100, "SPFC decomposition file must cover all 100 samples."

In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
cmd = [
    sys.executable,
    "scripts/bench_generate.py",
    "--manifest",
    manifest_path,
    "--run-root",
    RUN_ROOT,
    "--methods",
    METHOD,
    "--seed",
    str(SEED),
    "--guidance-scale",
    str(GUIDANCE_SCALE),
    "--skip-existing",
]

if METHOD == "spfc":
    cmd.extend(["--decompositions", REPO_DIR / DECOMP_REL])
if METHOD == "rectified_cfgpp":
    cmd.extend(["--rectified-repo-dir", RECTIFIED_REPO_DIR])

run_args(cmd, cwd=REPO_DIR)

In [ ]:
method_dir = RUN_ROOT / "t2i_compbench" / METHOD
index_path = method_dir / "index.json"
images = sorted(method_dir.glob("*.png"))
metadata = sorted(method_dir.glob("*.json"))
metadata = [path for path in metadata if path.name != "index.json"]

print("method_dir:", method_dir)
print("index:", index_path)
print("images:", len(images))
print("metadata files:", len(metadata))
assert index_path.exists(), f"Missing generation index: {index_path}"
assert len(images) == 100, f"Expected 100 images for {METHOD}, found {len(images)}."

readme = OUTPUT_ROOT / f"README_{METHOD}.txt"
readme.write_text(
    "T2I-CompBench 100 generated images\n"
    f"method: {METHOD}\n"
    f"seed: {SEED}\n"
    f"guidance_scale: {GUIDANCE_SCALE}\n"
    f"image_dir: {method_dir}\n",
    encoding="utf-8",
)
print("Kaggle output root:", OUTPUT_ROOT)
print("Attach this notebook output as a Kaggle dataset for the comparison notebook.")